In [1]:
import argparse
import json
import re
import sys
from pathlib import Path

# Allow imports from the project root (loaders/, inference/) regardless of
# which directory the script is called from.
# sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from loaders.dataloader import BioASQDataLoader
from loaders.base import BaseModelBackend
from loaders.local import VLLMBackend
from loaders.cloud import OpenRouterBackend


loader = BioASQDataLoader(
        path="/home/ucloud/BioASQ13B/data/val_data/13B1_golden_documents.jsonl",
        # limit=-1,
        # question_types= ["yesno", "factoid", "list", "summary"],
        # random_seed=42,
    )

/home/ucloud/BioASQ13B/phaseB/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:

def load_prompts(prompts_path: str) -> dict:
    with open(prompts_path) as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# Context building
# ---------------------------------------------------------------------------

def build_context(question: dict, input_type: str, num_support: int) -> str:
    if input_type == "snippets":
        items = question["snippets"][:num_support]
    else:
        items = [d["text"] for d in question["documents"][:num_support]]

    if not items:
        return "(No context available)"

    return "\n\n".join(f"{input_type}: {x}" for x in items)


# --------------------------------------

In [42]:
prompts_file = "/home/ucloud/BioASQ13B/phaseB/inference/prompts_generic.json"

if prompts_file is None:

    prompts_file = str(Path(__file__).parent / "prompts_generic.json")

num_support = "10"
prompt_ids = "7"
input_type = "abstract"
selected_counts: list[int]  = [int(n) for n in num_support.split(",")]
selected_prompts = [str(p) for p in prompt_ids.split(",")]

# Load data
loader = BioASQDataLoader(path="/home/ucloud/BioASQ13B/data/val_data/13B1_golden_documents.jsonl")
questions = list(loader)
print(f"Loaded {len(questions)} questions from {"/home/ucloud/BioASQ13B/data/val_data/13B1_golden_documents.jsonl"}")

# Load prompts
prompts_templates = load_prompts(prompts_file)

# Build all prompts: questions × num_support × prompt_ids
prompt_list: list[str] = []
prompt_info: list[tuple[str, int, str]] = []  # (qid, num_support, pid)

for q in questions:
    for n in selected_counts:
        context = build_context(q, input_type, n)
        for pid in selected_prompts:
            if pid not in prompts_templates:
                raise ValueError(f"Prompt id '{pid}' not found in {prompts_file}")
            template = prompts_templates[pid]["template"]
            prompt_list.append(template.format(
                d_type=input_type,
                context=context,
                question=q["body"],
            ))
            prompt_info.append((q["id"], n, pid))

print(f"Built {len(prompt_list)} prompts ({len(questions)} questions × {len(selected_counts)} counts × {len(selected_prompts)} prompt ids)")


Loaded 85 questions from /home/ucloud/BioASQ13B/data/val_data/13B1_golden_documents.jsonl
Built 85 prompts (85 questions × 1 counts × 1 prompt ids)


In [43]:
print(prompt_list[50])

Act as a biomedical expert. You will receive retrieved articles and snippets along with a question.

Your task is to draft an 'ideal answer' that approximates how a human expert would summarize the most relevant findings to answer the question.

<context>
abstract: Endometrioma patients are under-treated with endocrine endometriosis therapy.  STUDY QUESTION
Is there a difference in the use of endocrine endometriosis therapy in endometriosis patients with and without endometrioma?

SUMMARY ANSWER
Patients with endometriomas received significantly less endocrine endometriosis treatment (present intake in 42.5%) compared to patients with other forms of endometriosis and without endometriomas (present intake in 52.1%).

WHAT IS KNOWN ALREADY
Endocrine endometriosis therapy in patients with endometriomas reduces the risk of recurrence and therefore the risk of further surgery and subsequent irreversible damage to ovaries which results into reduced antral follicle counts (AFC), anti-Mulleria

In [21]:
prompt_list[25]

'Act as a biomedical expert. You will receive several abstracts summarizing research findings and methodologies. Along with this, a question will be provided. Your role is to analyze the abstracts and provide a scientifically accurate, concise answer to the question, leveraging the information from the abstracts.\n\nFirst read and understand the relevant information present in the several abstracts, extracting all relevant facts, explaining all your reasoning.\n\nAfter thinking about the information presented, present a final json containing your answer {"answer": answer}.\n\nAnswer in around 50 - 150 words, use a concise format with only plain text (no lists or markdown).\n\nQuestion: What is the role of the receptor tyrosine kinase AXL in malignancy?\n\nabstract: Giving AXL the axe: targeting AXL in human malignancy.  The receptor tyrosine kinase AXL, activated by a complex interaction between its ligand growth arrest-specific protein 6 and phosphatidylserine, regulates various vital

In [2]:
data = []
for i in loader:
    data.append(i)

In [32]:
data[49]

{'id': '67d3460218b1e36f2e000004',
 'body': 'Describe oncoEnrichR',
 'type': 'summary',
 'documents': [{'id': '37551617',
   'text': 'Comprehensive interrogation of gene lists from genome-scale cancer screens with oncoEnrichR.  Genome-scale screening experiments in cancer produce long lists of candidate genes that require extensive interpretation for biological insight and prioritization for follow-up studies. Interrogation of gene lists frequently represents a significant and time-consuming undertaking, in which experimental biologists typically combine results from a variety of bioinformatics resources in an attempt to portray and understand cancer relevance. As a means to simplify and strengthen the support for this endeavor, we have developed oncoEnrichR, a flexible bioinformatics tool that allows cancer researchers to comprehensively interrogate a given gene list along multiple facets of cancer relevance. oncoEnrichR differs from general gene set analysis frameworks through the in